# OBD Baseline Evaluation

This notebook computes baseline results for the OBD task on the same test split used by the training notebooks and compares them directly against the trained models.

What this notebook does:

1. Loads `data/obd_cot_gpt5.jsonl`
2. Rebuilds the same no-leak prompt and the same seeded test split
3. Runs text-only baseline inference on the OBD test set
4. Computes the same core metrics used in the trained-model notebooks
5. Adds `BERTScore` and embedding cosine similarity when the required packages are available
6. Compares baseline and trained systems in one aligned summary table


In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (CWD, *CWD.parents) if (path / 'pyproject.toml').exists()), CWD)
for extra_path in (PROJECT_ROOT, PROJECT_ROOT / 'src', PROJECT_ROOT / 'open_flamingo', PROJECT_ROOT / 'src' / 'open_flamingo'):
    if extra_path.exists() and str(extra_path) not in sys.path:
        sys.path.insert(0, str(extra_path))

from __future__ import annotations

import json
import os
import random
import re
import time
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any
from urllib import request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from IPython.display import display

from opentslm.system_metrics import SystemMetricsMonitor

try:
    from transformers import pipeline
except ImportError:
    pipeline = None

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


## Configuration

Choose the backend and the baseline models you want to run. The defaults keep the baseline aligned with the same Gemma and Llama backbones used in the trained experiments.


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'obd_cot_gpt5.jsonl'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'obd_baselines'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR = PROJECT_ROOT / 'results' / 'notebook_metrics'
METRICS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BACKEND = 'hf'  # 'hf', 'openai', or 'ollama'
MODEL_NAMES = [
    'google/gemma-3-270m',
    'meta-llama/Llama-3.2-1B',
]

MAX_SAMPLES = None  # set an int for a quick dry run
MAX_NEW_TOKENS = 220
TEMPERATURE = 0.1
SAVE_EVERY = 10
SLEEP_BETWEEN_CALLS = 0.0
REQUEST_TIMEOUT = 600

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
OPENAI_BASE_URL = os.getenv('OPENAI_BASE_URL', 'https://api.openai.com/v1/chat/completions')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')

FEATURE_LABELS = ['Speed', 'RPM', 'Engine Load', 'Throttle Position']
LABELS = ['economical', 'normal', 'aggressive']

TRAINED_SYSTEMS = [
    {
        'system_name': 'gemma_soft_prompt',
        'display_name': 'Gemma + Soft Prompt',
        'family': 'Gemma 3 270M',
        'group': 'trained',
        'summary_path': PROJECT_ROOT / 'data' / 'obd_eval_summary_gemma3_270m.json',
        'detailed_path': PROJECT_ROOT / 'data' / 'obd_eval_detailed_gemma3_270m.jsonl',
    },
    {
        'system_name': 'gemma_flamingo',
        'display_name': 'Gemma + Flamingo',
        'family': 'Gemma 3 270M',
        'group': 'trained',
        'summary_path': PROJECT_ROOT / 'data' / 'obd_alignment_test_summary_gemma3_270m.json',
        'detailed_path': PROJECT_ROOT / 'data' / 'obd_alignment_test_detailed_gemma3_270m.jsonl',
    },
    {
        'system_name': 'llama_soft_prompt',
        'display_name': 'Llama + Soft Prompt',
        'family': 'Llama 3.2 1B',
        'group': 'trained',
        'summary_path': PROJECT_ROOT / 'data' / 'obd_eval_summary_llama3_2_1B.json',
        'detailed_path': PROJECT_ROOT / 'data' / 'obd_eval_detailed_llama3_2_1B.jsonl',
    },
    {
        'system_name': 'llama_flamingo',
        'display_name': 'Llama + Flamingo',
        'family': 'Llama 3.2 1B',
        'group': 'trained',
        'summary_path': PROJECT_ROOT / 'data' / 'obd_alignment_test_summary_llama3_2_1B.json',
        'detailed_path': PROJECT_ROOT / 'data' / 'obd_alignment_test_detailed_llama3_2_1B.jsonl',
    },
]

_NO_LEAK_PROMPT = """You are shown a time-series plot of vehicle telemetry over a 120-sample window.
This data corresponds to one of two possible activities:
[{label}]
[{dis}]
Your task is to classify the activity based on analysis of the data.
Instructions:
- Begin by analyzing the time-series without assuming a specific label.
- Think step-by-step about what the observed patterns suggest regarding movement intensity and behavior.
- Write your rationale as a single, natural paragraph, do not use bullet points, numbered steps, or section headings.
- Do not refer back to the plot or to the act of visual analysis in your rationale; the plot is only for reference but you should reason about the time-series data.
- Do not assume any answer at the beginning, analyze as if you do not yet know which class is correct.
- Do not mention either class label until the final sentence.
- End your response with exactly "Answer: <label>" where <label> is the correct class."""

print('Data path:', DATA_PATH)
print('Output dir:', OUTPUT_DIR)
print('Metrics dir:', METRICS_DIR)
print('Backend:', BACKEND)
print('Models:', MODEL_NAMES)
for item in TRAINED_SYSTEMS:
    print(item['display_name'], '->', 'FOUND' if item['summary_path'].exists() else 'MISSING')


## Load Data and Recreate the Same Test Split


In [ ]:
rows = [json.loads(x) for x in DATA_PATH.read_text(encoding='utf-8').splitlines() if x.strip()]
rows = [r for r in rows if r.get('label') != 'congested']

def build_no_leak_prompt(row: dict[str, Any]) -> str:
    return _NO_LEAK_PROMPT.format(label=row.get('label') or '', dis=row.get('dissimilar_label') or '')

for row in rows:
    row['pre_prompt'] = build_no_leak_prompt(row)

idx = list(range(len(rows)))
random.shuffle(idx)

n = len(idx)
train_end = max(1, int(0.6 * n))
val_end = max(train_end + 1, int(0.8 * n)) if n > 2 else train_end

train_rows = [rows[i] for i in idx[:train_end]]
val_rows = [rows[i] for i in idx[train_end:val_end]]
test_rows = [rows[i] for i in idx[val_end:]]

if len(val_rows) == 0 and len(train_rows) > 1:
    val_rows = [train_rows.pop()]
if len(test_rows) == 0 and len(train_rows) > 1:
    test_rows = [train_rows.pop()]

if MAX_SAMPLES is not None:
    test_rows = test_rows[:MAX_SAMPLES]

print('train/val/test:', len(train_rows), len(val_rows), len(test_rows))
print('Example test id:', test_rows[0]['id'] if test_rows else 'N/A')


## Build the Text-Only OBD Input

This follows the same textual signal summarization used in the OBD soft-prompt notebook so the baseline remains comparable.


In [ ]:
def summarize_series(label: str, values: torch.Tensor, n_points: int = 8) -> str:
    values = values.detach().cpu().float()
    n = values.numel()
    idx = torch.linspace(0, max(n - 1, 0), steps=min(n_points, n)).long()
    samples = ', '.join(f'{values[i].item():.1f}' for i in idx)
    delta = values[-1].item() - values[0].item()
    if delta > 1e-3:
        trend = 'overall increasing'
    elif delta < -1e-3:
        trend = 'overall decreasing'
    else:
        trend = 'roughly flat'
    return (
        f'{label} series with mean {values.mean().item():.2f}, '
        f'std {values.std(unbiased=False).item():.2f}, '
        f'min {values.min().item():.2f}, max {values.max().item():.2f}, '
        f'trend {trend}, sampled points [{samples}]:'
    )

def build_input_text(row: dict[str, Any]) -> str:
    ts = torch.tensor(row['time_series'], dtype=torch.float32)
    if ts.ndim != 2 or ts.shape[0] < 4:
        raise ValueError(f"Unexpected time_series shape for id={row.get('id')}: {tuple(ts.shape)}")
    summaries = [
        summarize_series(label, series)
        for label, series in zip(FEATURE_LABELS, [ts[0], ts[1], ts[2], ts[3]])
    ]
    return f"{row['pre_prompt']} | {' | '.join(summaries)} | {row.get('post_prompt', 'Rationale:')}"

test_examples = [
    {
        'id': row['id'],
        'input_text': build_input_text(row),
        'target': row['answer'],
        'reference_label': row.get('label'),
    }
    for row in test_rows
]

print('Built test examples:', len(test_examples))
print(test_examples[0]['input_text'][:800] if test_examples else 'No test examples')


## Inference Helpers


In [ ]:
def model_suffix(model_name: str) -> str:
    suffix = model_name.split('/')[-1].replace(':', '_').replace('-', '_').replace('.', '_')
    return suffix.lower()

def post_json(url: str, payload: dict[str, Any], headers: dict[str, str] | None = None) -> dict[str, Any]:
    body = json.dumps(payload).encode('utf-8')
    req = request.Request(url, data=body, headers={**(headers or {}), 'Content-Type': 'application/json'})
    with request.urlopen(req, timeout=REQUEST_TIMEOUT) as response:
        return json.loads(response.read().decode('utf-8'))

def generate_with_openai(model_name: str, prompt: str) -> str:
    if not OPENAI_API_KEY:
        raise ValueError('OPENAI_API_KEY is not set.')
    payload = {
        'model': model_name,
        'temperature': TEMPERATURE,
        'messages': [{'role': 'user', 'content': prompt}],
        'max_tokens': MAX_NEW_TOKENS,
    }
    headers = {'Authorization': f'Bearer {OPENAI_API_KEY}'}
    data = post_json(OPENAI_BASE_URL, payload, headers=headers)
    return data['choices'][0]['message']['content'].strip()

def generate_with_ollama(model_name: str, prompt: str) -> str:
    payload = {
        'model': model_name,
        'stream': False,
        'messages': [{'role': 'user', 'content': prompt}],
        'options': {'temperature': TEMPERATURE, 'num_predict': MAX_NEW_TOKENS},
    }
    data = post_json(f'{OLLAMA_BASE_URL}/api/chat', payload)
    return data['message']['content'].strip()

def load_hf_pipeline(model_name: str):
    if pipeline is None:
        raise ImportError('transformers is not installed in this environment.')
    return pipeline(
        task='text-generation',
        model=model_name,
        torch_dtype='auto',
        device_map='auto',
        temperature=TEMPERATURE,
    )

def generate_with_hf(pipe, prompt: str) -> str:
    outputs = pipe(prompt, max_new_tokens=MAX_NEW_TOKENS, return_full_text=False)
    if not outputs:
        return ''
    return outputs[0]['generated_text'].strip()


## Metrics

The functions below keep the baseline aligned with the trained-model notebooks. They compute the same core metrics and then add optional semantic metrics when the required libraries are available.


In [ ]:
def normalize_tokens(text: str) -> list[str]:
    return re.findall(r'[a-z0-9_\-]+', str(text).lower())

def token_f1(pred: str, gold: str) -> float:
    p = normalize_tokens(pred)
    g = normalize_tokens(gold)
    if not p or not g:
        return 0.0
    common = Counter(p) & Counter(g)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(g)
    return 2 * precision * recall / (precision + recall)

def rouge_l_f1(pred: str, gold: str) -> float:
    p = normalize_tokens(pred)
    g = normalize_tokens(gold)
    if not p or not g:
        return 0.0
    dp = [[0] * (len(g) + 1) for _ in range(len(p) + 1)]
    for i in range(1, len(p) + 1):
        for j in range(1, len(g) + 1):
            if p[i - 1] == g[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    lcs = dp[-1][-1]
    if lcs == 0:
        return 0.0
    precision = lcs / len(p)
    recall = lcs / len(g)
    return 2 * precision * recall / (precision + recall)

def exact_match(pred: str, gold: str) -> int:
    return int(str(pred).strip().lower() == str(gold).strip().lower())

def seq_sim(pred: str, gold: str) -> float:
    return SequenceMatcher(None, str(pred).strip().lower(), str(gold).strip().lower()).ratio()

def extract_label(text: str | None) -> str | None:
    if not text:
        return None
    t = str(text).strip().lower()
    m = re.findall(r'answer\s*:\s*([a-z_\-]+)', t)
    if m:
        cand = m[-1]
        return cand if cand in LABELS else None
    positions = [(t.rfind(label), label) for label in LABELS if t.rfind(label) != -1]
    if positions:
        positions.sort()
        return positions[-1][1]
    return None

def classification_metrics(predictions: list[dict[str, Any]]) -> tuple[float, float, list[dict[str, Any]]]:
    rows = []
    y_true, y_pred = [], []
    for item in predictions:
        gt = extract_label(item.get('target'))
        pr = extract_label(item.get('prediction'))
        rows.append({'id': item['id'], 'gt_label': gt, 'pred_label': pr, 'accuracy': int(gt is not None and pr == gt)})
        if gt is not None and pr is not None:
            y_true.append(gt)
            y_pred.append(pr)

    total = len(y_true)
    correct = sum(1 for g, p in zip(y_true, y_pred) if g == p)
    accuracy = (correct / total) if total else 0.0

    macro_f1_sum = 0.0
    for label in LABELS:
        tp = sum(1 for g, p in zip(y_true, y_pred) if g == label and p == label)
        fp = sum(1 for g, p in zip(y_true, y_pred) if g != label and p == label)
        fn = sum(1 for g, p in zip(y_true, y_pred) if g == label and p != label)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
        macro_f1_sum += f1
    macro_f1 = macro_f1_sum / len(LABELS) if LABELS else 0.0
    return accuracy, macro_f1, rows

def compute_advanced_metrics(predictions: list[dict[str, Any]], detailed_rows: list[dict[str, Any]]) -> dict[str, Any]:
    advanced_metrics: dict[str, Any] = {}
    pred_texts = [item['prediction'] for item in predictions]
    tgt_texts = [item['target'] for item in predictions]

    try:
        from bert_score import score as bertscore_score
        _, _, bert_f1 = bertscore_score(pred_texts, tgt_texts, lang='en', verbose=False)
        bert_values = bert_f1.detach().cpu().numpy().tolist()
        advanced_metrics['bertscore_f1_mean'] = float(np.mean(bert_values)) if bert_values else 0.0
        for row, val in zip(detailed_rows, bert_values):
            row['bertscore_f1'] = float(val)
    except Exception as exc:
        advanced_metrics['bertscore_f1_mean'] = None
        advanced_metrics['bertscore_error'] = str(exc)

    try:
        from sentence_transformers import SentenceTransformer
        emb_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
        pred_emb = emb_model.encode(pred_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        tgt_emb = emb_model.encode(tgt_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        cosine_scores = (pred_emb * tgt_emb).sum(axis=1).tolist()
        advanced_metrics['embedding_cosine_mean'] = float(np.mean(cosine_scores)) if cosine_scores else 0.0
        for row, val in zip(detailed_rows, cosine_scores):
            row['embedding_cosine'] = float(val)
    except Exception as exc:
        advanced_metrics['embedding_cosine_mean'] = None
        advanced_metrics['embedding_cosine_error'] = str(exc)

    return advanced_metrics

def build_metrics(predictions: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    detailed_rows = []
    token_scores, rouge_scores, em_scores, seq_scores = [], [], [], []

    for item in predictions:
        tok = token_f1(item['prediction'], item['target'])
        rou = rouge_l_f1(item['prediction'], item['target'])
        em = exact_match(item['prediction'], item['target'])
        sim = seq_sim(item['prediction'], item['target'])
        pred_label = extract_label(item['prediction'])
        pred_text = str(item.get('prediction', ''))
        has_answer_tag = bool(re.search(r'answer\s*:', pred_text.lower()))
        non_ascii_ratio = (sum(ord(ch) > 127 for ch in pred_text) / max(len(pred_text), 1))

        row = dict(item)
        row.update({
            'token_f1': float(tok),
            'rouge_l_f1': float(rou),
            'exact_match': int(em),
            'seq_sim': float(sim),
            'pred_label': pred_label,
            'valid_label': pred_label in LABELS,
            'has_answer_tag': has_answer_tag,
            'prediction_length': len(pred_text),
            'non_ascii_ratio': float(non_ascii_ratio),
        })
        detailed_rows.append(row)
        token_scores.append(tok)
        rouge_scores.append(rou)
        em_scores.append(em)
        seq_scores.append(sim)

    accuracy, macro_f1, cls_rows = classification_metrics(predictions)
    cls_by_id = {r['id']: r for r in cls_rows}
    for row in detailed_rows:
        row.update(cls_by_id[row['id']])

    advanced_metrics = compute_advanced_metrics(predictions, detailed_rows)

    summary = {
        'n': len(predictions),
        'token_f1_mean': float(np.mean(token_scores)) if token_scores else 0.0,
        'rouge_l_f1_mean': float(np.mean(rouge_scores)) if rouge_scores else 0.0,
        'exact_match_mean': float(np.mean(em_scores)) if em_scores else 0.0,
        'seq_sim_mean': float(np.mean(seq_scores)) if seq_scores else 0.0,
        'accuracy': float(accuracy),
        'macro_f1': float(macro_f1),
        'validity_metrics': {
            'has_answer_tag_rate': float(np.mean([r['has_answer_tag'] for r in detailed_rows])) if detailed_rows else 0.0,
            'valid_label_rate': float(np.mean([r['valid_label'] for r in detailed_rows])) if detailed_rows else 0.0,
            'mean_prediction_length': float(np.mean([r['prediction_length'] for r in detailed_rows])) if detailed_rows else 0.0,
            'mean_non_ascii_ratio': float(np.mean([r['non_ascii_ratio'] for r in detailed_rows])) if detailed_rows else 0.0,
        },
        'advanced_metrics': advanced_metrics,
    }
    return detailed_rows, summary


## Run Baselines on the Test Set

This loop supports resume-by-file so you can stop and continue later without losing completed predictions. Metrics are recomputed at the end so the saved summaries always stay aligned with the current evaluation protocol.


In [ ]:
all_model_summaries = []
baseline_artifacts = []

for model_name in MODEL_NAMES:
    suffix = model_suffix(model_name)
    display_name = f"{model_name.split('/')[-1]} baseline"
    out_pred = OUTPUT_DIR / f'obd_baseline_predictions_{suffix}.jsonl'
    out_detail = OUTPUT_DIR / f'obd_baseline_detailed_{suffix}.jsonl'
    out_summary = OUTPUT_DIR / f'obd_baseline_summary_{suffix}.json'
    metrics_samples_path = METRICS_DIR / f'obd_baseline_{suffix}_samples.csv'
    metrics_summary_path = METRICS_DIR / f'obd_baseline_{suffix}_summary.json'

    predictions = []
    completed_ids = set()
    if out_pred.exists():
        with out_pred.open('r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                predictions.append(row)
                completed_ids.add(row['id'])
        print(f'Resuming {model_name} from {len(predictions)} saved predictions.')

    pending_examples = [x for x in test_examples if x['id'] not in completed_ids]
    print(f'Running {model_name} on {len(pending_examples)} pending test examples...')

    model_monitor = SystemMetricsMonitor(
        label=f'obd_baseline_{suffix}',
        interval_s=0.5,
        disk_path=PROJECT_ROOT,
        metadata={'notebook': 'obd_baseline_evaluation', 'stage': 'inference', 'model_name': model_name, 'backend': BACKEND},
    ).start()
    model_monitor.mark('setup', pending_examples=len(pending_examples), resumed_examples=len(predictions))

    hf_pipe = None
    if BACKEND == 'hf' and pending_examples:
        pipeline_start = time.perf_counter()
        hf_pipe = load_hf_pipeline(model_name)
        model_monitor.mark('pipeline_loaded', pipeline_load_time_s=time.perf_counter() - pipeline_start)

    for idx, example in enumerate(tqdm(pending_examples, desc=f'{model_name}', unit='sample'), start=1):
        started = time.perf_counter()
        if BACKEND == 'hf':
            prediction = generate_with_hf(hf_pipe, example['input_text'])
        elif BACKEND == 'openai':
            prediction = generate_with_openai(model_name, example['input_text'])
        elif BACKEND == 'ollama':
            prediction = generate_with_ollama(model_name, example['input_text'])
        else:
            raise ValueError(f'Unsupported BACKEND: {BACKEND}')
        latency = time.perf_counter() - started

        predictions.append({
            'id': example['id'],
            'prompt': example['input_text'],
            'prediction': prediction,
            'target': example['target'],
            'model_name': model_name,
            'backend': BACKEND,
        })
        model_monitor.mark(
            'inference_step',
            step=idx,
            inference_latency_s=latency,
            prediction_chars=len(prediction),
            sample_id=example['id'],
        )

        if idx % SAVE_EVERY == 0 or idx == len(pending_examples):
            with out_pred.open('w', encoding='utf-8') as f:
                for row in predictions:
                    f.write(json.dumps(row, ensure_ascii=False) + '\n')

        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)

    detailed_rows, summary = build_metrics(predictions)
    summary.update({
        'model_name': model_name,
        'backend': BACKEND,
        'system_name': f'{suffix}_baseline',
        'display_name': display_name,
        'family': 'Gemma 3 270M' if 'gemma' in model_name.lower() else 'Llama 3.2 1B',
        'group': 'baseline',
    })

    with out_detail.open('w', encoding='utf-8') as f:
        for row in detailed_rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    out_summary.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    model_monitor.stop(final_phase='finished', n_predictions=len(predictions), pending_examples=len(pending_examples))
    model_monitor.to_csv(metrics_samples_path)
    metrics_summary_path.write_text(json.dumps(model_monitor.summary().to_dict(), ensure_ascii=False, indent=2), encoding='utf-8')

    all_model_summaries.append(summary)
    baseline_artifacts.append({
        'system_name': summary['system_name'],
        'display_name': summary['display_name'],
        'family': summary['family'],
        'group': summary['group'],
        'summary_path': out_summary,
        'detailed_path': out_detail,
    })
    print(f'Saved predictions to {out_pred}')
    print(f'Saved detailed metrics to {out_detail}')
    print(f'Saved summary to {out_summary}')
    print(f'Saved metrics samples to {metrics_samples_path}')
    print(f'Saved metrics summary to {metrics_summary_path}')
    print(summary)


## Compare Baselines and Trained Models

This section flattens the summary structure into one table so the baseline can be compared fairly against the four trained systems.


In [ ]:
def read_jsonl_rows(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def flatten_summary(summary: dict[str, Any], default_group: str, default_display_name: str | None = None, default_family: str | None = None) -> dict[str, Any]:
    validity = summary.get('validity_metrics', {}) or {}
    advanced = summary.get('advanced_metrics', {}) or {}
    row = {
        'system_name': summary.get('system_name', default_display_name or summary.get('model_name', 'unknown')),
        'display_name': summary.get('display_name', default_display_name or summary.get('model_name', 'unknown')),
        'group': summary.get('group', default_group),
        'family': summary.get('family', default_family),
        'model_name': summary.get('model_name'),
        'backend': summary.get('backend'),
        'n': summary.get('n', summary.get('n_samples')),
        'token_f1_mean': summary.get('token_f1_mean'),
        'rouge_l_f1_mean': summary.get('rouge_l_f1_mean'),
        'exact_match_mean': summary.get('exact_match_mean'),
        'seq_sim_mean': summary.get('seq_sim_mean'),
        'accuracy': summary.get('accuracy'),
        'macro_f1': summary.get('macro_f1'),
        'has_answer_tag_rate': validity.get('has_answer_tag_rate', summary.get('has_answer_tag_rate')),
        'valid_label_rate': validity.get('valid_label_rate', summary.get('valid_label_rate')),
        'mean_prediction_length': validity.get('mean_prediction_length', summary.get('mean_prediction_length')),
        'mean_non_ascii_ratio': validity.get('mean_non_ascii_ratio', summary.get('mean_non_ascii_ratio')),
        'bertscore_f1_mean': advanced.get('bertscore_f1_mean'),
        'embedding_cosine_mean': advanced.get('embedding_cosine_mean'),
    }
    return row

baseline_rows = [flatten_summary(summary, default_group='baseline') for summary in all_model_summaries]
trained_rows = []
trained_artifacts = []
for item in TRAINED_SYSTEMS:
    summary = json.loads(item['summary_path'].read_text(encoding='utf-8'))
    summary.update({
        'system_name': item['system_name'],
        'display_name': item['display_name'],
        'family': item['family'],
        'group': item['group'],
    })
    trained_rows.append(flatten_summary(summary, default_group='trained', default_display_name=item['display_name'], default_family=item['family']))
    trained_artifacts.append({
        'system_name': item['system_name'],
        'display_name': item['display_name'],
        'family': item['family'],
        'group': item['group'],
        'summary_path': item['summary_path'],
        'detailed_path': item['detailed_path'],
    })

comparison_df = pd.DataFrame(baseline_rows + trained_rows)
comparison_df['group'] = pd.Categorical(comparison_df['group'], categories=['baseline', 'trained'], ordered=True)
comparison_df = comparison_df.sort_values(['family', 'group', 'accuracy', 'macro_f1', 'token_f1_mean'], ascending=[True, True, False, False, False]).reset_index(drop=True)

display(comparison_df)

comparison_path = OUTPUT_DIR / 'obd_baseline_vs_trained_comparison.csv'
comparison_df.to_csv(comparison_path, index=False)
print('Saved comparison table to', comparison_path)


comparison_artifacts = baseline_artifacts + trained_artifacts


## Focused Per-Family Comparison

These tables make the fair comparison explicit: each baseline is compared first against the trained variants that share the same backbone family.


In [ ]:
for family in sorted(comparison_df['family'].dropna().unique()):
    print(f'=== {family} ===')
    display(
        comparison_df.loc[comparison_df['family'] == family, [
            'display_name', 'group', 'token_f1_mean', 'rouge_l_f1_mean', 'accuracy', 'macro_f1',
            'bertscore_f1_mean', 'embedding_cosine_mean', 'seq_sim_mean', 'valid_label_rate',
            'has_answer_tag_rate', 'mean_prediction_length'
        ]].reset_index(drop=True)
    )


## Notes

- The baseline is a text-only direct prompting baseline over the same telemetry summaries used in the OBD soft-prompt notebook.
- The split, prompt construction, and core metrics are aligned with the trained-model notebooks so the comparison makes sense.
- `BERTScore` and embedding cosine are computed only when the required packages are installed; otherwise the notebook records the error but still keeps the table aligned.
- The fairest reading is usually per backbone family: Gemma baseline vs Gemma trained variants, and Llama baseline vs Llama trained variants.


## Visual Comparison

These charts highlight complementary views: aggregate metric ranking, backbone-specific improvement from baseline to trained methods, and per-sample metric distributions.


In [ ]:

metric_plot_df = comparison_df.copy()
metric_plot_df['display_name'] = pd.Categorical(metric_plot_df['display_name'], categories=comparison_df['display_name'].tolist(), ordered=True)

plot_metrics = [
    ('accuracy', 'Accuracy'),
    ('macro_f1', 'Macro-F1'),
    ('token_f1_mean', 'Token-F1'),
    ('rouge_l_f1_mean', 'ROUGE-L F1'),
    ('bertscore_f1_mean', 'BERTScore F1'),
    ('embedding_cosine_mean', 'Embedding Cosine'),
]

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
for ax, (metric, title) in zip(axes.flat, plot_metrics):
    plot_df = metric_plot_df.dropna(subset=[metric]).copy()
    sns.scatterplot(
        data=plot_df,
        x=metric,
        y='display_name',
        hue='group',
        style='family',
        s=180,
        palette={'baseline': '#7f7f7f', 'trained': '#1f77b4'},
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel(title)
    ax.set_ylabel('')
    if ax is not axes.flat[0]:
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()

handles, labels = axes.flat[0].get_legend_handles_labels()
axes.flat[0].legend(handles, labels, title='Type / Family', loc='best')
plt.tight_layout()
plt.show()


In [ ]:

slope_rows = []
for family in sorted(comparison_df['family'].dropna().unique()):
    fam = comparison_df[comparison_df['family'] == family].copy()
    baseline = fam[fam['group'] == 'baseline'].sort_values(['accuracy', 'macro_f1', 'token_f1_mean'], ascending=False).head(1)
    trained = fam[fam['group'] == 'trained'].copy()
    if baseline.empty or trained.empty:
        continue
    fam_rows = pd.concat([baseline, trained], ignore_index=True)
    order_map = {}
    for _, row in fam_rows.iterrows():
        name = row['display_name']
        lower = str(name).lower()
        if 'baseline' in lower:
            order_map[name] = 0
        elif 'soft prompt' in lower:
            order_map[name] = 1
        elif 'flamingo' in lower:
            order_map[name] = 2
        else:
            order_map[name] = 99
    fam_rows['stage_order'] = fam_rows['display_name'].map(order_map)
    fam_rows = fam_rows.sort_values('stage_order')
    for metric in ['accuracy', 'macro_f1', 'bertscore_f1_mean', 'embedding_cosine_mean']:
        for _, row in fam_rows.dropna(subset=[metric]).iterrows():
            slope_rows.append({
                'family': family,
                'display_name': row['display_name'],
                'stage_order': row['stage_order'],
                'stage_label': row['display_name'].replace(' baseline', ''),
                'metric': metric,
                'value': row[metric],
            })

slope_df = pd.DataFrame(slope_rows)
metric_titles = {
    'accuracy': 'Accuracy',
    'macro_f1': 'Macro-F1',
    'bertscore_f1_mean': 'BERTScore F1',
    'embedding_cosine_mean': 'Embedding Cosine',
}

fig, axes = plt.subplots(2, 2, figsize=(20, 12), sharex=False)
for ax, metric in zip(axes.flat, ['accuracy', 'macro_f1', 'bertscore_f1_mean', 'embedding_cosine_mean']):
    subset = slope_df[slope_df['metric'] == metric].copy()
    if subset.empty:
        ax.set_visible(False)
        continue
    for family, fam_df in subset.groupby('family'):
        fam_df = fam_df.sort_values('stage_order')
        ax.plot(fam_df['stage_order'], fam_df['value'], marker='o', linewidth=3, label=family)
        for _, row in fam_df.iterrows():
            ax.text(row['stage_order'], row['value'], row['stage_label'], fontsize=9, ha='center', va='bottom')
    ax.set_title(f'Slope chart: {metric_titles[metric]}')
    ax.set_xlabel('System progression within backbone family')
    ax.set_ylabel(metric_titles[metric])
    ax.set_xticks([0, 1, 2], ['Baseline', 'Soft Prompt', 'Flamingo'])
    ax.legend(title='Family')

plt.tight_layout()
plt.show()


In [ ]:

detailed_frames = []
for item in comparison_artifacts:
    path = item.get('detailed_path')
    if path is None or not Path(path).exists():
        continue
    rows = read_jsonl_rows(Path(path))
    if not rows:
        continue
    df = pd.DataFrame(rows)
    df['display_name'] = item['display_name']
    df['family'] = item['family']
    df['group'] = item['group']
    detailed_frames.append(df)

detailed_compare_df = pd.concat(detailed_frames, ignore_index=True) if detailed_frames else pd.DataFrame()

if detailed_compare_df.empty:
    print('No detailed rows found for per-sample plots.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(24, 7))
    per_sample_metrics = [
        ('token_f1', 'Token-F1 per sample'),
        ('bertscore_f1', 'BERTScore F1 per sample'),
        ('embedding_cosine', 'Embedding cosine per sample'),
    ]
    for ax, (metric, title) in zip(axes, per_sample_metrics):
        plot_df = detailed_compare_df.dropna(subset=[metric]).copy()
        if plot_df.empty:
            ax.set_visible(False)
            continue
        sns.boxplot(data=plot_df, x='family', y=metric, hue='group', ax=ax)
        sns.stripplot(data=plot_df, x='family', y=metric, hue='group', dodge=True, alpha=0.35, size=3, ax=ax)
        ax.set_title(title)
        ax.set_xlabel('Backbone family')
        ax.set_ylabel(metric)
        handles, labels = ax.get_legend_handles_labels()
        unique = []
        seen = set()
        for h, l in zip(handles, labels):
            if l not in seen:
                seen.add(l)
                unique.append((h, l))
        ax.legend([h for h, _ in unique[:2]], [l for _, l in unique[:2]], title='Group')
    plt.tight_layout()
    plt.show()
